# Part 1 round 2: measured development improvement

File-backed CPU analysis of already-completed Colab runs; no new GPU inference.
Separate two comparisons: round 1 vs round 2 format stability, and dense-only vs
final retrieval within each round. Earlier cells preserve the pre-review snapshot; completed human metrics are appended below. Changes followed
inspection of recorded failures, so this is development evidence, not a holdout.


In [1]:
import sys,json,hashlib,subprocess
from pathlib import Path
import pandas as pd
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'eval/part1/results/round2_colab_20261114T032206Z/summary.json').is_file())
commit=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip()
print('Analysis commit:',commit)
print('Analysis Python:',sys.version.split()[0],'pandas:',pd.__version__)
print('Execution: CPU analysis of saved records; source inference: Colab Tesla T4; seed: 266')
print('Original Colab source commit: unavailable, not retroactively assigned')
paths=[ROOT/'eval/part1/results/round1_colab_before_schema_fix/summary.json',ROOT/'eval/part1/results/round2_colab_20261114T032206Z/summary.json']
rounds=[json.loads(p.read_text(encoding='utf-8')) for p in paths]
assert rounds[0]['questions_specification_sha256']==rounds[1]['questions_specification_sha256']
print('Frozen specification:',rounds[0]['questions_specification_sha256'])


Analysis commit: 9caac5f34d2b843c5aaf5968eccd379ac2074c40
Analysis Python: 3.11.9 pandas: 3.1.1
Execution: CPU analysis of saved records; source inference: Colab Tesla T4; seed: 266
Original Colab source commit: unavailable, not retroactively assigned
Frozen specification: e4cf051593f3665cec1e411b5b745c64e33b92e9369d25c508f71e6fa01e1763


In [2]:
from IPython.display import display,Markdown
rows=[]
for run in rounds:
    for cohort in ['practice','dense_only','final']:
        r=run['practice'] if cohort=='practice' else run['golden'][cohort]
        rows.append(dict(round=run['round'],cohort=cohort,questions=r.get('questions',r.get('denominator')),
            execution_errors=r['execution_errors'],format_fallbacks=r['format_fallbacks'],
            median_seconds_per_question=r['median_seconds_per_question'],
            recall_at_5=r.get('retrieval_recall_at_5'),retrieval_denominator=r.get('retrieval_denominator'),
            answer_correctness=r.get('answer_correctness'),citation_accuracy=r.get('citation_accuracy'),
            groundedness_mean=r.get('groundedness_mean'),human_agreement=r.get('groundedness_agree_within_one')))
table=pd.DataFrame(rows)
display(table)
changes=[]
for cohort in ['practice','dense_only','final']:
    before,after=[r for r in rows if r['cohort']==cohort]
    changes.append(dict(cohort=cohort,before_fallbacks=before['format_fallbacks'],after_fallbacks=after['format_fallbacks'],
        fewer_fallbacks=before['format_fallbacks']-after['format_fallbacks'],
        reduction_percentage_points=100*(before['format_fallbacks']-after['format_fallbacks'])/after['questions']))
display(pd.DataFrame(changes))
print('Format improvements do not establish answer correctness or a higher Kaggle score.')


Format improvements do not establish answer correctness or a higher Kaggle score.


,round,cohort,questions,execution_errors,format_fallbacks,median_seconds_per_question,recall_at_5,retrieval_denominator,answer_correctness,citation_accuracy,groundedness_mean,human_agreement
0,1,practice,40,0,4,5.729000,NaN,NaN,None,None,None,None
1,1,dense_only,25,0,4,12.840790,0.472222,24.0,None,None,None,None
2,1,final,25,0,6,0.270933,0.763889,24.0,None,None,None,None
3,2,practice,40,0,0,5.126000,NaN,NaN,None,None,None,None
4,2,dense_only,25,0,2,9.173569,0.472222,24.0,None,None,None,None
5,2,final,25,0,1,0.795780,0.763889,24.0,None,None,None,None


,cohort,before_fallbacks,after_fallbacks,fewer_fallbacks,reduction_percentage_points
0,practice,4,0,4,10.0
1,dense_only,4,2,2,8.0
2,final,6,1,5,20.0


In [3]:
report='# Part 1 round 2: notebook-computed development comparison\n\n'
report+='Analysis commit: `'+commit+'`. Source GPU runs used one Tesla T4; analysis here is CPU/file-backed. Original inference Git commit was unavailable.\n\n'
report+='| Cohort | Round 1 fallbacks | Round 2 fallbacks | Fewer fallback percentage points |\n| --- | ---: | ---: | ---: |\n'
for r in changes:report+=f"| {r['cohort']} | {r['before_fallbacks']} | {r['after_fallbacks']} | {r['reduction_percentage_points']:.1f} |\n"
report+='\nWithin-round second-run required comparison:\n\n| Metric | Dense-only baseline | Final system |\n| --- | ---: | ---: |\n'
a,b=rounds[1]['golden']['dense_only'],rounds[1]['golden']['final']
for label,key in [('Questions','denominator'),('Recall@5','retrieval_recall_at_5'),('Recall denominator','retrieval_denominator'),('Median seconds/request','median_seconds_per_question'),('Answer correctness','answer_correctness'),('Citation accuracy','citation_accuracy'),('Groundedness mean','groundedness_mean'),('Human agreement within one point','groundedness_agree_within_one')]:
    fmt=lambda v:'Pending' if v is None else f'{v:.4f}' if isinstance(v,float) else str(v)
    report+=f'| {label} | {fmt(a[key])} | {fmt(b[key])} |\n'
report+='\nThis documents a measured format-stability improvement. It does not establish better answer correctness, citation accuracy or Kaggle score. Recall is unchanged between rounds. Final-system latency increased between rounds; fewer fallbacks may allow longer answers, but causal attribution is not established. Short final-system medians include deterministic structured routes, not just VLM inference.\n'
report+='\nKnown content failure: P1V2-22 reverses the correction scope. DOC3:p6 corrects the plotting error for 2012; other values remain the same. Both members must independently rate groundedness, and content/citation judgments remain pending for both systems.\n'
report+='\nFixed questions, rubrics, model revisions and seed were retained. These are development reruns after inspecting failures; do not claim an untouched holdout. The source repository is currently public; course privacy, final Git provenance, report, human review, final tag and Kaggle selection remain separate completion items.\n'
target=ROOT/'eval/part1/results/round2_colab_20261114T032206Z'
(target/'ROUND2_IMPROVEMENT.md').write_text(report,encoding='utf-8')
(target/'round_comparison.json').write_text(json.dumps(dict(rows=rows,changes=changes,analysis_commit=commit,source_inference_commit=None),indent=2),encoding='utf-8')
display(Markdown(report))


# Part 1 round 2: notebook-computed development comparison

Analysis commit: `9caac5f34d2b843c5aaf5968eccd379ac2074c40`. Source GPU runs used one Tesla T4; analysis here is CPU/file-backed. Original inference Git commit was unavailable.

| Cohort | Round 1 fallbacks | Round 2 fallbacks | Fewer fallback percentage points |
| --- | ---: | ---: | ---: |
| practice | 4 | 0 | 10.0 |
| dense_only | 4 | 2 | 8.0 |
| final | 6 | 1 | 20.0 |

Within-round second-run required comparison:

| Metric | Dense-only baseline | Final system |
| --- | ---: | ---: |
| Questions | 25 | 25 |
| Recall@5 | 0.4722 | 0.7639 |
| Recall denominator | 24 | 24 |
| Median seconds/request | 9.1736 | 0.7958 |
| Answer correctness | Pending | Pending |
| Citation accuracy | Pending | Pending |
| Groundedness mean | Pending | Pending |
| Human agreement within one point | Pending | Pending |

This documents a measured format-stability improvement. It does not establish better answer correctness, citation accuracy or Kaggle score. Recall is unchanged between rounds. Final-system latency increased between rounds; fewer fallbacks may allow longer answers, but causal attribution is not established. Short final-system medians include deterministic structured routes, not just VLM inference.

Known content failure: P1V2-22 reverses the correction scope. DOC3:p6 corrects the plotting error for 2012; other values remain the same. Both members must independently rate groundedness, and content/citation judgments remain pending for both systems.

Fixed questions, rubrics, model revisions and seed were retained. These are development reruns after inspecting failures; do not claim an untouched holdout. The source repository is currently public; course privacy, final Git provenance, report, human review, final tag and Kaggle selection remain separate completion items.


## Post-run human groundedness review (2026-11-14)

The supplied 25-question ratings were bound to the saved second-round final-system run hash. B originally contained the wrong reviewer name; the user explicitly confirmed Pratiksha Kaushik independently rated it. Original files and scores are preserved, with the correction documented only in derived records.

| Measurement | Round 2 final system |
| --- | ---: |
| Yuyao Ding mean | 4.44 / 5 |
| Pratiksha Kaushik mean | 4.44 / 5 |
| Combined groundedness mean | 4.44 / 5 |
| Agreement within one point | 25/25 (100%) |
| Exact agreement | 23/25 (92%) |

This is CPU aggregation of existing ratings, with no new inference. It does not supply answer-correctness, citation-support, dense-only or earlier-run human scores. The existing executed cells and outputs above are retained.

[Original ratings, identity correction, metrics and limitations](../../eval/part1/results/round2_colab_20261114T032206Z/human_review_20261114/README.md).


## Completed source-content and groundedness review (2026-11-15 PT)

The next code cell was executed in a **separate CPU-only Python 3.12.2 kernel**
against saved predictions and supplied ratings; its execution count starts at 1.
Earlier cells and outputs preserve the original pre-review analysis. No GPU
inference is repeated. The source review is by Yuyao Ding; the existing independent
groundedness scores are by Yuyao Ding and Pratiksha Kaushik. One baseline label was
corrected only after explicit user confirmation, with original bytes preserved.
Numerical meaning is judged by source review against the unchanged frozen rubric;
the older token-presence heuristic is separately reported, including its inability
to match a target magnitude 1.6 to the correctly signed decrease -1.6.


In [1]:
from pathlib import Path
import json, sys, subprocess, hashlib
from IPython.display import display, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "eval/part1/results/round2_colab_20261114T032206Z/content_review_20261115/part1-content-review.original.json").is_file())
sys.path.insert(0, str(ROOT / "scripts"))
from finalize_p1_content_review import aggregate, read, digest
RESULTS = ROOT / "eval/part1/results/round2_colab_20261114T032206Z"
D = RESULTS / "content_review_20261115"
corrections = read(D / "corrections.json")
corrections["verified_source_review_sha256"] = digest(D / "part1-content-review.original.json")
groundedness = read(RESULTS / "human_review_20261114/ratings_and_metrics.json")
outcome, effective = aggregate(RESULTS, ROOT / "eval/p1_golden_v2.json", read(D / "part1-content-review.original.json"), corrections, groundedness)
assert outcome == read(D / "metrics.json")
assert effective == read(D / "effective_review.json")
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
print("Analysis commit:", commit)
print("Analysis Python:", sys.version.split()[0])
print("Execution: file-backed CPU aggregation only; no model, training or inference")
print("Inference source: second recorded Colab Tesla T4 run; original inference commit unavailable")
print("Original content review SHA256:", digest(D / "part1-content-review.original.json"))
print("Content reviewer:", outcome["content_reviewer"])
print("Groundedness reviewers:", ", ".join(outcome["groundedness_reviewers"]))
a, b = outcome["metrics"]
rows = [
    ("Answer correctness", f"{a['answer_correct_out_of_25']}/25 ({100*a['answer_correctness']:.2f}%)", f"{b['answer_correct_out_of_25']}/25 ({100*b['answer_correctness']:.2f}%)"),
    ("Retrieval Recall@5 (24 evidence-bearing cases)", f"{a['retrieval_recall_at_5']:.3f}", f"{b['retrieval_recall_at_5']:.3f}"),
    ("Citation accuracy", f"{a['supported_citations']}/{a['citation_denominator']} ({100*a['citation_accuracy']:.2f}%)", f"{b['supported_citations']}/{b['citation_denominator']} ({100*b['citation_accuracy']:.2f}%)"),
    ("Two-member groundedness, mean (1-5)", "Not collected for baseline", f"{b['groundedness_mean']:.2f}"),
    ("Groundedness agreement within one point", "Not collected for baseline", f"{round(25*b['groundedness_agree_within_one'])}/25 ({100*b['groundedness_agree_within_one']:.0f}%)"),
    ("Median seconds/question", f"{a['median_seconds_per_question']:.4f}", f"{b['median_seconds_per_question']:.4f}")
]
table = "| Measurement | Dense-only baseline | Final system |\n| --- | ---: | ---: |\n" + "\n".join("| " + " | ".join(row) + " |" for row in rows)
display(Markdown(table))
remaining = [r["question_id"] for r in effective["modes"]["final"]["records"] if not r["answer_correct"]]
print("Final answers judged incorrect:", ", ".join(remaining))
discrepancies = [r for r in outcome["records"] if r["numeric_token_discrepancy_requires_context"]]
print("Legacy numeric-token diagnostics (not overrides of source-content review):")
print(json.dumps(discrepancies, indent=2))
text = """# Part 1 round 2: completed source-content review and results

This table was generated by a separately executed CPU notebook analysis of saved
Colab Tesla T4 predictions and submitted human ratings. No inference was repeated.
The original question set, reference values, generated answers, source records
and the two independent groundedness files remain unchanged.

""" + table + """

## Review provenance and confirmed correction

Yuyao Ding supplied correctness labels for both sets of answers and support labels
for every emitted page citation. The original file is preserved byte-for-byte as
`part1-content-review.original.json`. The user explicitly confirmed that baseline
P1V2-11 should be incorrect; the original true label remains in that file and the
separate `corrections.json` records its authority, reason and effective false label.
`effective_review.json` is a derived copy, not a replacement original.

The two-member groundedness scores are reused from the previous completed review;
no baseline groundedness was collected or inferred. Validation checks source-file
hashes, row identities, emitted citation sets, run identities and score types. It
does not independently establish how the reviewer conducted the source review.

## Numerical scoring boundary

Correctness uses supplied human source-content judgments of the frozen required
facts, numerical values within the existing ±1% tolerance, units and direction,
plus the exact expected status. The previous numeric-token helper is retained
only as a diagnostic: final P1V2-09 expresses a decline of 1.6 as a signed change
of -1.6. Its literal target-token check fails despite the numerical meaning
matching the reference. This is disclosed rather than silently dropping that
answer. No numeric target, reference, inference rule or human label was changed
for this diagnostic. Baseline P1V2-11 is different: the answer is substantively
incomplete/wrong and the correction was explicitly confirmed by the user.

This differs from blindly applying the legacy token-presence gate; consumers
must use this disclosed source-content scoring method with these review results.

## Interpretation and limits

The final system has higher reviewed answer correctness, retrieval recall and
citation support in this fixed development evaluation, yet some answers and
many emitted citations still fail review. Groundedness, answer correctness and
citation support are distinct metrics with different units and denominators.

The final answers judged incorrect are: """ + ", ".join(remaining) + """.
The review notes identify forecast/outturn confusion, slope/intercept confusion,
missing inflation-series details, unemployment/participation comparison errors,
a fabricated revision and an incomplete erratum answer. Preserve these failures.

The final latency includes structured-evidence and session-memory answers that
avoid a VLM call; it is not model-only decoding speed. Questions/rubrics were
fixed, but development fixes were informed by prior failures. These scores are
not an independent holdout result, a Kaggle score, or a guarantee of hidden-set
accuracy. The final tagged Kaggle execution/selection remains a separate task.

See `provenance.json` for input hashes and `metrics.json` for all per-case counts
and the retained token-check diagnostic. The earlier `summary.json` and the
original groundedness aggregation retain their historical null fields; this
completed record supplements them rather than rewriting original observations.
"""
_ = (D / "RESULTS.md").write_text(text, encoding="utf-8")


Analysis commit: b3d953332c34bfa57a5596b0e2c62ebfd000bc74
Analysis Python: 3.12.2
Execution: file-backed CPU aggregation only; no model, training or inference
Inference source: second recorded Colab Tesla T4 run; original inference commit unavailable
Original content review SHA256: 2d3394a882ba5249e22b49e582be7b244bd606cbca1f92725fb6f0b514e05e26
Content reviewer: Yuyao Ding
Groundedness reviewers: Yuyao Ding, Pratiksha Kaushik
Final answers judged incorrect: P1V2-06, P1V2-10, P1V2-11, P1V2-13, P1V2-20, P1V2-22
Legacy numeric-token diagnostics (not overrides of source-content review):
[
  {
    "mode": "final",
    "question_id": "P1V2-09",
    "human_answer_correct": true,
    "status_matches": true,
    "counted_correct": true,
    "legacy_numeric_token_check": false,
    "numeric_token_discrepancy_requires_context": true,
    "supported_citations": 1,
    "emitted_citations": 5
  }
]


| Measurement | Dense-only baseline | Final system |
| --- | ---: | ---: |
| Answer correctness | 7/25 (28.00%) | 19/25 (76.00%) |
| Retrieval Recall@5 (24 evidence-bearing cases) | 0.472 | 0.764 |
| Citation accuracy | 14/75 (18.67%) | 28/47 (59.57%) |
| Two-member groundedness, mean (1-5) | Not collected for baseline | 4.44 |
| Groundedness agreement within one point | Not collected for baseline | 25/25 (100%) |
| Median seconds/question | 9.1736 | 0.7958 |

The computed table and detailed provenance are also saved in [the completed review results](../../eval/part1/results/round2_colab_20261114T032206Z/content_review_20261115/RESULTS.md).
